In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import scanpy as sc
import pandas as pd
import subprocess
from rich import print
import glob
import matplotlib.image as mpimg

In [ ]:
import sys
sys.path.append('/Volumes/B/Projects/2025Project/SpatialAnalysis/Code')
import SpatialPyALab as lab   

In [ ]:
data_dir = '/Volumes/B/Projects/2025Project/SpatialAnalysis/Project_ICC_spatial_V2/Data/SpatialData/0_Samples/'
sample_id = os.path.basename(os.getcwd())
root_dir = data_dir+sample_id
root_dir

In [ ]:
h5ad_file = os.path.join(root_dir,'preprocess/mask_tissue.h5ad')
adata_spot = sc.read_h5ad(h5ad_file)

In [ ]:
out_path=os.path.join(root_dir,"ficture/CellType")
os.makedirs(out_path, exist_ok=True)
ref_path="/Volumes/B/Projects/2025Project/SpatialAnalysis/Code/FictureByLab/Data/ICC_reference"

In [ ]:
coor_key = 'spatial_manual_fine'
mu_scale = adata_spot.uns['mu_scale'][coor_key]
log_path = os.path.join(out_path, "run_ficture_scRNA.log")   
anno_df = pd.read_csv(os.path.join(ref_path,'annotation.tsv'), sep='\t',header=None)
whiteLiet = os.path.join(ref_path,'Whitelist_Gene.tsv')
nFactor = len(anno_df)

In [ ]:
ficture_path = "/Volumes/B/Projects/2025Project/SpatialAnalysis/Code/FictureByLab"
conda_env = "/Users/yanru/miniforge3/envs/ficture"

cmd = [ "conda", "run", "-p", conda_env,
       "snakemake",
       "--snakefile", "./scripts/snakemake/ficture.smk",
       "--cores", "36",
       "--keep-going",
       "--rerun-incomplete",
       "--printshellcmds",
       "--config",
       f"h5ad_file={h5ad_file}",
       f"reference_path={ref_path}",
       f"root_dir={out_path}",
       f"genes_file={whiteLiet}",
       f"dummy_genes=^RPL|^RPS|^HB[ABDEGMQZ]|^MT-|ENSG|LINC|HLA-|IG[HKL]|SLC35E2B|NCAM2|GABRB3|KIAA1958|CYP1D1P|PLVAP|MS4A19P|TRPM3|UBE2D3|SERPINA1|PLEKHN1|RBBP8NL|SPP1|EEF1A1|ALB",
       f"nFactor={nFactor}",
       f"fit_width=8",
       f"anchor_res=3",
       f"min_ct_per_unit=20",#20
       f"min_ct_per_fit=10",#10
       f"coord_key={coor_key}",
       f"mu_scale={mu_scale}",
       "pseudo=True",
       "figure_bg=ffffff",

       "target=run_ficture",
       ]

env = os.environ.copy()
env["MPLBACKEND"] = "Agg"

with open(log_path, "w") as log_file:
    subprocess.run(cmd, cwd=ficture_path, env=env, stdout=log_file, stderr=subprocess.STDOUT)

In [ ]:
class_file = os.path.join(out_path,'analysis/barcode/match_barcode_cluster.tsv.gz')
anno_file=os.path.join(ref_path,'annotation.tsv')
adata_spot = lab.annotation.merge_ficture_result(adata = adata_spot, anno_name = 'CellType_ficture',
                                                 class_file = class_file,
                                                 anno_file = anno_file)

In [ ]:
os.makedirs(os.path.join(root_dir,'ficture'), exist_ok=True)
adata_spot.write(os.path.join(root_dir,f"ficture/{sample_id}_anno_ficture.h5ad"))